# Pattern Recognition 2 - Feature Space

**Objectives**

* Compute distances between objects in feature space
* Use PCA to perform reduce the dimensionality of the problem
* Implement face recognition using [Eigenfaces](http://en.wikipedia.org/wiki/Eigenface)

## Feature space distance

The `ocr_features.npy` file contains extracted features for each letter from the document used in the previous lab, and the `ocr_class.npy` file contains the character corresponding to these features.

The 18 features, extracted with the `regionprops` methods from `scikit-image` are, in order: 

> area, convex area, eccentricity, equivalent diameter, extent, filled area, intertia tensor eigenvalue 1, inertia tensor eigenvalue 2, major axis length, minor axis length, Hu moment 1, Hu moment 2, Hu moment 3, Hu moment 4, Hu moment 5, Hu moment 6, perimeter, solidity

* Create a method which computes the Euclidian distance, in feature space, between two objects.
* Check if the "nearest" object is of the same class.
* What can you do to improve this "distance-based" classification? 

In [2]:
import numpy as np

ocr_features = np.load('../data/ocr_features.npy')
ocr_class = np.load('../data/ocr_class.npy')

print(ocr_features.shape)
print(np.unique(ocr_class))

FileNotFoundError: [Errno 2] No such file or directory: '../data/ocr_features.npy'

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

ocr_features = (ocr_features-ocr_features.mean(axis=0))/ocr_features.std(axis=0)

plt.figure(figsize=(15, 5))
plt.boxplot(ocr_features)
plt.show()

In [ ]:
## -- Your code here -- ##

def euclidian(A, B):
    return np.sqrt(((A-B)**2).sum())

best_matches = []
for i in range(ocr_features.shape[0]):
    best_match = -1
    best_d = 100000000
    for j in range(ocr_features.shape[0]):
        if i != j:
            d = euclidian(ocr_features[i], ocr_features[j])
            if d < best_d:
                best_match = j
                best_d = d
    best_matches.append((i, ocr_class[i], best_match, ocr_class[best_match], best_d))

In [ ]:
best_matches = []
for i in range(ocr_features.shape[0]):
    distances = np.sqrt(((ocr_features-ocr_features[i])**2).sum(axis=1))
    distances[i] = distances.max()
    best_match = distances.argmin()
    best_matches.append((i, ocr_class[i], best_match, ocr_class[best_match], distances.min()))

In [ ]:
best_matches = np.array(best_matches)
acc = (best_matches[:, 1] == best_matches[:, 3]).sum()/best_matches.shape[0]
print(acc)

In [ ]:
for matches in best_matches:
    if matches[1] != matches[3]:
        print(matches[1], matches[3])

## Eigenfaces

The `faces.npy` file contains a 3D matrix containing 2963 images, each with 170x200 pixels, encoded in 8-bit grayscale.

In [ ]:
from matplotlib import pyplot as plt
%matplotlib notebook

faces = np.load('../../extern_data/faces.npy')
print(faces.shape, faces.dtype, faces.min(), faces.max())

In [ ]:
# Show some faces:
plt.figure()
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(faces[i], cmap=plt.cm.gray)
    plt.axis('off')
plt.show()

### Direct approach

1. Write a program that subsample the images (e.g. by 5), converts 2D images into lines, and collect all these lines into one single matrix **T**.
1. Zero-center **T** by removing the "mean image" ($T_i \leftarrow T_i - T_{mean}$)
1. Compute the variance/covariance matrix of **T**:
$$ \mathbf{S} = \mathbf{T^TT}$$
1. Compute the eigenvalues $\lambda_i$ and eigenvectors $\mathbf{v}_i$ of **S**
$$\mathbf{Sv}_i = \mathbf{T^T}\mathbf{Tv}_i = \lambda_i \mathbf{v}_i$$
1. The eigenvectors have the same size as the images, and are often referred to as "eigenfaces". Display some of them.

In [ ]:
# Example of eigen value extraction
import numpy as np
from numpy import linalg as LA

w,v = LA.eig(np.diag((1, 2, 3)))
print(w,v)

In [ ]:
# Example of subsampling
im = faces[0]
plt.figure()
plt.subplot(1,2,1)
plt.imshow(im, cmap=plt.cm.gray)
plt.title("Original image")
plt.subplot(1,2,2)
plt.imshow(im[::5,::5], cmap=plt.cm.gray)
plt.title("Subsampled image")
plt.show()

In [ ]:
## -- Your code here -- ##

T = np.array([face[::5,::5].flatten() for face in faces])
print(T.shape)

plt.gray()
im_mean = T.mean(axis=0)
print(im_mean.shape)

T = T-im_mean

plt.figure()
plt.imshow(im_mean.reshape((200//5, 170//5)))
plt.show()

plt.figure(figsize=(10, 15))
plt.imshow(T)
plt.show()

print(T.mean(axis=0))

In [ ]:
S = np.dot(T.T, T)
print(S.shape)

In [ ]:
w, v = LA.eig(S)
print(w.shape, v.shape)

In [ ]:
order = w.argsort()[::-1]
w = w[order]
v = v[:,order]

In [ ]:
explained_variance = w/w.sum()
for i in range(len(explained_variance)):
    if explained_variance[:i].sum() > 0.9:
        print(i, explained_variance[:i].sum())
        break
for i in range(len(explained_variance)):
    if explained_variance[:i].sum() > 0.95:
        print(i, explained_variance[:i].sum())
        break

In [ ]:
plt.figure()
plt.imshow(v[:,0].reshape((40, 34)))

### Compression

1. Reconstruct images using only the first N eigenfaces (e.g. N=100).
2. Compare (visually) reconstructed images with original images

In [ ]:
## -- Your code here -- ##

def encode(im, n):
    return np.dot(im, v)[:n]

def decode(encoded, n):
    return np.dot(encoded, (v.T)[:n])

face = T[12]
n = 100
encoded = encode(face, n)
decoded = decode(encoded, n)
plt.figure()
plt.imshow((face+im_mean).reshape((40,34)))
plt.figure()
plt.imshow((decoded+im_mean).reshape((40,34)))
plt.show()


### Face recognition

1. Using the simplified vector space (e.g. 100 first eigenfaces), compute the euclidian distance between one face and the others.
2. For a subset of the image of the database, find the 4 closest matches.

In [ ]:
## -- Your code here -- ##

N = 250
T_weights = np.array([encode(face, N) for face in T])
print(T_weights.shape)

In [ ]:
face_id = 1

distances = np.sqrt(((T_weights - T_weights[face_id])**2).sum(axis=1))
distances[face_id] = distances.max()
best_matches = np.argsort(distances)

plt.figure(figsize=(15,5))
plt.subplot(1, 6, 1)
plt.imshow(T[face_id].reshape((40,34)))
for i in range(5):
    plt.subplot(1, 6, i+2)
    plt.imshow(T[best_matches[i]].reshape((40,34)))
    plt.title(distances[i])
plt.show()

### Indirect approach

Instead of using the eigenvalues/eigenvectors of $\mathbf{S} = \mathbf{T^TT}$, compute the the eigenvalues/eigenvectors of $\mathbf{Q} = \mathbf{TT^T}$.

Let $\mathbf{u_i}$ be the eigenvectors of $\mathbf{Q}$. We have:
$$\mathbf{TT^T}\mathbf{u}_i = \lambda_i \mathbf{u}_i$$

By multiplying to the left with $\mathbf{T^T}$, we have:
$$\mathbf{T^T}\mathbf{TT^T}\mathbf{u}_i = \lambda_i\mathbf{T^T}\mathbf{u}_i$$
$$\mathbf{S}\mathbf{T^T}\mathbf{u}_i = \lambda_i\mathbf{T^T}\mathbf{u}_i$$

Which means that if $\mathbf{u}_i$ is eigenvector of $\mathbf{Q}$, then $\mathbf{T^T}\mathbf{u}_i$ is eigenvector of $\mathbf{S}$.

The indirect approach thus becomes:
1. Compute the eigenvectors $\mathbf{u}_i$ of $\mathbf{Q} = \mathbf{TT^T}$.
1. Compute the eigenfaces with $\mathbf{v}_i = \mathbf{T^T}\mathbf{u}_i$ 
1. Reconstruct the images using only the first N eigenfaces (e.g. N=100)
1. Compare (visually) reconstructed images with original images
1. Use the indirect approach with more resolution (without subsampling the images)

Use this new feature space to perform the same face recognition method.

In [ ]:

## -- Your code here -- ##
